---
title: "3.4 Saving figures for a paper or slides"
---

::: {.lesson-tools}
[Open in Colab](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/lessons/03-viz/04-saving-figures.ipynb){.btn .btn-primary}
[Download notebook](../../downloads/03-viz/04-saving-figures.ipynb){.btn .btn-outline-secondary}
:::

**Run in your browser.** Colab needs a Google account to run and save work. If its GitHub link cannot open, download this notebook and use **File → Upload notebook** at [Colab](https://colab.research.google.com/). You can read the website without an account. A notebook is a document containing explanatory text and runnable code cells.

**Time:** about 35 minutes, including exercises · **Before you begin:** Complete [3.3 Comparing groups and small multiples](../03-viz/03-groups-small-multiples.ipynb) first. Each notebook starts afresh: it does not remember variables from another lesson.

## You will learn

- Save a named figure as a raster image and a vector PDF.
- Choose physical size, resolution and background deliberately.
- Preserve a caption, source and accessible text alongside an exported figure.

## Why it matters

A chart that looks clear inside a notebook can become unreadable in a paper or slide. A **raster** image stores pixels; PNG is a common example. A **vector** file stores drawing instructions, so lines and text can stay sharp when resized; PDF can contain vector figures. Saving changes the output format, not the evidence in a chart. We practise with made-up numbers, then export the running-example scatter reproducibly.

## Worked example

### Keep the figure handle

Create a small fictional line chart and retain its Figure as `practice_fig`. A named handle identifies exactly which figure you will save; it avoids relying on whichever figure happens to be current. The numbers are made-up Northland values.

In [ ]:
#| label: fig-saving-practice
#| fig-alt: "Made-up Northland life expectancy rises across three labelled years, on a six-by-four-inch figure."
#| fig-cap: "Made-up numbers for a fictional country."
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 12})
practice = pd.DataFrame({"year": [2000, 2010, 2020], "life_exp": [68.0, 71.0, 74.0]})
practice_fig, practice_ax = plt.subplots(figsize=(6, 4), layout="constrained")
practice_ax.plot(practice["year"], practice["life_exp"], marker="o", color="#21645a")
practice_ax.set(xlabel="Year", ylabel="Life expectancy (years)", title="Northland · made-up numbers")
print("Size in inches:", practice_fig.get_size_inches())
plt.show()

The physical size is six by four inches. **DPI** means dots per inch: for a PNG saved at 300 DPI, that nominal canvas has 1800 × 1200 pixels. Increasing DPI does not fix labels that are too small for the final displayed size.

### Prepare the real country panel

Run this prepared setup once in a fresh session. It uses the course CSV in a local checkout, or downloads its fixed, checksum-verified public source in Colab. A checksum is a fingerprint of a file's contents; a mismatch stops the download from being used. You do not need to retype the setup. It does not use your Google Drive or any private data.

In [ ]:
# Prepared setup: use the course CSV locally; fetch its pinned source in Colab.
from pathlib import Path
import hashlib
import io
from urllib.request import urlopen
import pandas as pd

data_file = Path("data/gapminder.csv")
if not data_file.exists():
    source = "https://raw.githubusercontent.com/jennybc/gapminder/391b5a40d3574c3b89d889dd7d72f7f3fdb4a70c/inst/extdata/gapminder.tsv"
    with urlopen(source, timeout=30) as response:
        raw = response.read()
    expected = "f74275b4a2f4a7c8e3a09a752f9d6e43d1759d78d5cbfdebaaa3e1a6efc68f78"
    if hashlib.sha256(raw).hexdigest() != expected:
        raise ValueError("The data download changed. Please report this to the course maintainer.")
    downloaded = pd.read_csv(io.BytesIO(raw), sep="\t", dtype=str)
    downloaded = downloaded.rename(columns={"continent": "region", "lifeExp": "life_exp", "pop": "population", "gdpPercap": "gdp_pc"})
    data_file.parent.mkdir(parents=True, exist_ok=True)
    downloaded.to_csv(data_file, index=False, lineterminator="\n")
print("Course data ready: data/gapminder.csv")

Look for **Course data ready**. Colab working files are temporary; rerun this cell after a runtime reset. The data cover 142 countries at five-year intervals from 1952 to 2007. Each row is one country in one year.

### Build the figure you intend to deliver

Recreate the real 2007 log-income scatter at the desired final size. Set an explicit white figure background for use in a paper and include the measurement units. Keep a caption separately; a PNG file does not automatically carry the web page's alt text or source note.

In [ ]:
#| label: fig-export-ready
#| fig-alt: "Export-ready 2007 scatter of 142 countries, showing a broad positive association between log-scaled GDP per person and life expectancy with substantial dispersion."
#| fig-cap: "Historical Gapminder teaching extract via Jennifer Bryan (CC0). Country observations receive equal weight unless stated otherwise."
countries = pd.read_csv(data_file)
year_2007 = countries.loc[countries["year"] == 2007].copy()
export_fig, export_ax = plt.subplots(figsize=(6, 4), layout="constrained", facecolor="white")
export_ax.scatter(year_2007["gdp_pc"], year_2007["life_exp"], color="#21645a", alpha=0.8, edgecolor="white", linewidth=0.4)
export_ax.set_xscale("log")
export_ax.set(xlabel="GDP per person (inflation-adjusted US dollars; log axis)", ylabel="Life expectancy at birth (years)", title="Income and life expectancy · 2007")
export_ax.grid(axis="y", alpha=0.25)
plt.show()

The figure is the same descriptive cross-section you built earlier. It does not acquire stronger causal meaning when exported. Saving from the retained Figure works even after display; using a generic save command after creating another figure can save the wrong image.

### Save PNG and PDF from the same object

`Path` joins folder and file names. The generated download folder is temporary and ignored by Git. `savefig` chooses a format from the file ending. Save the same six-by-four-inch Figure as a 300-DPI PNG and a vector PDF. The PDF timestamp fields are omitted to avoid unnecessary date differences between builds.

In [ ]:
output_dir = Path("downloads/outputs")
output_dir.mkdir(parents=True, exist_ok=True)
png_file = output_dir / "income-life-2007.png"
pdf_file = output_dir / "income-life-2007.pdf"
export_fig.savefig(png_file, dpi=300, facecolor="white")
export_fig.savefig(pdf_file, facecolor="white", metadata={"Title": "Income and life expectancy, 2007", "CreationDate": None, "ModDate": None})
print(png_file, png_file.stat().st_size, "bytes")
print(pdf_file, pdf_file.stat().st_size, "bytes")

Both files should exist and have nonzero sizes. Bytes measure file size, not visual quality. On this website download the [300-DPI PNG](../../downloads/outputs/income-life-2007.png) or [PDF](../../downloads/outputs/income-life-2007.pdf). In Colab, use the left Files panel, open downloads → outputs, and download the generated files; these website links do not refer to Colab's temporary filesystem.

### Save the explanation too

Store a caption and a plain-language image description alongside the files. A caption documents source, sample and units. **Alt text** communicates essential visual information to a reader who cannot see an image. Add appropriate alt text in the document or slides where the image will actually be used; a separate text file is a useful handoff, not automatic accessibility compliance.

In [ ]:
figure_note = """Caption: 142 countries in 2007, one equally weighted country observation per point. GDP per person is in the historical extract's inflation-adjusted US dollars on a log axis; life expectancy at birth is in years. Source: Gapminder via Jennifer Bryan, CC0. This association does not establish a causal income effect.
Alt text: A country scatter shows life expectancy generally rising with income per person, with substantial variation at similar incomes. The log income axis uses equal spacing for equal ratios.
"""
notes_file = output_dir / "income-life-2007-notes.txt"
notes_file.write_text(figure_note, encoding="utf-8")
print(figure_note)

The [notes file](../../downloads/outputs/income-life-2007-notes.txt) travels with the image. Original course figures still have undecided licensing; attributing the CC0 underlying data does not itself select a license for the course's original presentation. Verify rights before reusing material beyond the permissions granted.

## Try it

Work in your own notebook copy. Add a new code cell for each task, try it yourself, then open the solution to compare. Run from the top if a name is missing.

### 1. Export a smaller PNG

Save the same figure at 150 DPI under a new filename. State the expected pixel dimensions.

<details><summary>Solution 1</summary>

Run this complete solution after the worked example.

In [ ]:
small_png = output_dir / "income-life-2007-150dpi.png"
export_fig.savefig(small_png, dpi=150, facecolor="white")
print("Expected pixels:", 6 * 150, "by", 4 * 150)

The nominal canvas is 900 × 600 pixels. The website also offers [the smaller PNG](../../downloads/outputs/income-life-2007-150dpi.png). Check legibility at the size where you will place it; resizing a low-resolution image upward cannot recover missing detail.

</details>

### 2. Check what was saved

Read the first eight bytes of the PNG and the first five bytes of the PDF. Then print whether the captions file exists.

<details><summary>Solution 2</summary>

Run this complete solution after the worked example.

In [ ]:
print(png_file.read_bytes()[:8])
print(pdf_file.read_bytes()[:5])
print(notes_file.exists())

The headers identify a PNG and a PDF, and the notes file exists. A header check alone does not establish that labels are readable or the intended observations were plotted; the course also checks image size and inspects the rendered chart.

</details>

### 3. Preserve the source sample

Count the plotted 2007 rows and verify that the raw panel still has its six source columns. Did saving modify the CSV?

<details><summary>Solution 3</summary>

Run this complete solution after the worked example.

In [ ]:
print(len(year_2007))
print(countries.shape)
print(export_ax.get_xscale())

The results are 142, (1704, 6) and log. Exporting the Figure changes no raw measurements; preserving the original file is a separate workflow responsibility.

</details>

## When it goes wrong

The following examples deliberately trigger specific errors. The prepared `try`/`except` wrapper catches that named error and prints its message so the notebook can continue. It is for these demonstrations; an unexpected error elsewhere should still stop execution. You can read the message without memorizing the wrapper.

### An output folder does not exist

An empty temporary directory makes the missing location predictable without touching another project's files.

In [ ]:
try:
    from tempfile import TemporaryDirectory
    with TemporaryDirectory() as temporary:
        export_fig.savefig(Path(temporary) / "absent-folder" / "chart.png")
except FileNotFoundError as error:
    print(f"{type(error).__name__}: {error}")

Create the intended parent folder first with mkdir(parents=True, exist_ok=True). Do not scatter outputs into arbitrary personal folders while troubleshooting.

### An unsupported file ending is used

The file ending is how savefig normally chooses a format.

In [ ]:
try:
    export_fig.savefig(output_dir / "chart.not_a_format")
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

Choose a supported format such as .png or .pdf, or an explicit supported format argument. Renaming a file ending after saving does not convert its contents.

## Recap

- Save a named Figure rather than an unspecified current image.
- Physical size and DPI jointly determine PNG pixel dimensions.
- Inspect labels at the intended final size; vector output is not a cure for poor design.
- Keep caption, alt text, source and rights information with each export.

**Next:** [4.1 Descriptive statistics and a Table 1](../04-regression/01-descriptive-statistics.ipynb).